# Imports 

In [1]:
import joblib

import numpy as np
import pandas as pd

from fastapi import FastAPI
import httpx
import json

from IPython.display import display

In [2]:
%load_ext autoreload
%autoreload 2

# Repertoire de travail

In [3]:
%pwd

'/home/user_daren_mabanza/data_science/projets/V2_projet_fraud_detection/02_notebooks'

In [4]:
%cd "/home/user_daren_mabanza/data_science/projets/V2_projet_fraud_detection"

/home/user_daren_mabanza/data_science/projets/V2_projet_fraud_detection


# Import des données 

In [5]:
model = joblib.load("06_models/FullXGBoostModel_V1.joblib")
cost_optimizer = joblib.load("07_artifacts/FullCostOptimizer.joblib")

In [6]:
cost_optimizer.get_best_threshold()

0.16080402010050251

In [10]:
transactions = [
    {"amt": 52.80, "heure": 18, "jour": "Monday",    "category": "grocery_pos",   "state": "CA"},
    {"amt": 65.40, "heure": 8,  "jour": "Tuesday",   "category": "gas_transport", "state": "TX"},
    {"amt": 28.50, "heure": 12, "jour": "Wednesday", "category": "food_dining",   "state": "NY"},
    {"amt": 149.99,"heure": 21, "jour": "Thursday",  "category": "shopping_net",  "state": "FL"},
    {"amt": 45.00, "heure": 20, "jour": "Friday",    "category": "entertainment", "state": "IL"},
    {"amt": 89.90, "heure": 15, "jour": "Saturday",  "category": "shopping_pos",  "state": "WA"},
    {"amt": 12.75, "heure": 10, "jour": "Sunday",    "category": "personal_care", "state": "AZ"},
    {"amt": 350.00,"heure": 14, "jour": "Monday",    "category": "travel",        "state": "CO"},
    {"amt": 950.00,"heure": 2,  "jour": "Saturday",  "category": "shopping_net",  "state": "CA"},
    {"amt": 1.50,  "heure": 23, "jour": "Sunday",    "category": "misc_net",      "state": "NY"},
]

# Construction de l'API

In [84]:
def predict(data:list[dict]):
    
    observations = pd.DataFrame([data])
    
    probabilite = model.predict_proba(observations)[:,1]
    prediction = (probabilite > cost_optimizer.get_best_threshold()).astype(int)
    
    probabilite = round(float(probabilite[0])*100,2)
    prediction = int(prediction[0])

    
    if prediction == 1:
        return {
            "message" : "La transaction a été refusé",
            "prédiction" : prediction,
            "probabilité" : probabilite,
        }
        
    else :
        return {
            "message" : "La transaction a été accepté",
            "prédiction" : prediction,
            "probabilité" : probabilite, 
        }

In [85]:
predict(transactions[8])

{'message': 'La transaction a été refusé',
 'prédiction': 1,
 'probabilité': 81.63}

In [ ]:
app = FastAPI(
    title = "API - Detection des transactions frauduleuses",
    version = "1.0.0",
)

@app.get("/health")
def health():
    return {"status" : "ok"}

@app.post("/predict")
def predict(data:list[dict]):
    
    observations = pd.DataFrame([data])
    
    probabilite = model.predict_proba(observations)[:,1]
    prediction = (probabilite > cost_optimizer.get_best_threshold()).astype(int)
    
    probabilite = round(float(probabilite[0])*100,2)
    prediction = int(prediction[0])

    
    if prediction == 1:
        return {
            "message" : "La transaction a été refusé",
            "prédiction" : prediction,
            "probabilité" : probabilite,
        }
        
    else :
        return {
            "message" : "La transaction a été accepté",
            "prédiction" : prediction,
            "probabilité" : probabilite, 
        }
        
@app.get("/model-info")
def model_info():
    return{"model" : "FullXGBoostModel"}

# Test API

In [58]:
body = {"amt": 950, "heure": 22,  "jour": "Monday",  "category": "shopping_net",  "state": "WY"}

In [59]:
reponse = httpx.post(
    "http://127.0.0.1:8000/predict",
    json = body,
    timeout = 5.0,
)

display(reponse.status_code)
display(reponse.content.decode("utf-8"))
display(reponse.url)

200

'{"message":"La transaction a été refusé","prédiction":1,"probabilité":78.57}'

URL('http://127.0.0.1:8000/predict')

In [19]:
resultat = json.loads(reponse.content.decode("utf-8"))
print(resultat.get("message"))
print(resultat.get("probabilité"))
print(resultat.get("prédiction"))

La transaction a été refusé
81.63
1
